# sse

> Server-Sent Events parsing utilities used by streaming transports.

In [ ]:
#| default_exp sse


## Notebook Overview

This notebook documents `fastllm_v2.sse` in nbdev style, interleaving explanations, source cells, and lightweight REPL checks.

- **Depends on:** none (leaf module)
- **Primary symbols:** `SSEvent, parse_sse_lines, aiter_sse`


## Module Setup

Imports, constants, and shared setup used by the symbols below.


In [ ]:
#| export
"SSE parsing helpers."

from __future__ import annotations

from dataclasses import dataclass
from typing import AsyncIterator, Iterator, Optional

import httpx


### `SSEvent`

A parsed SSE event.


In [ ]:
#| export


@dataclass(frozen=True)
class SSEvent:
    "A parsed SSE event."
    event: Optional[str]
    data: str
    id: Optional[str] = None
    retry: Optional[int] = None


### `parse_sse_lines`

Parse SSE line stream into events.


In [ ]:
#| export


def parse_sse_lines(lines: Iterator[str]) -> Iterator[SSEvent]:
    "Parse SSE line stream into events."
    event, data, event_id, retry = None, [], None, None
    for raw in lines:
        line = raw.rstrip("\n")
        if not line:
            if data: yield SSEvent(event=event, data="\n".join(data), id=event_id, retry=retry)
            event, data, event_id, retry = None, [], None, None
            continue
        if line.startswith(":"): continue
        field, _, value = line.partition(":")
        if value.startswith(" "): value = value[1:]
        if field == "event": event = value
        elif field == "data": data.append(value)
        elif field == "id": event_id = value
        elif field == "retry":
            try: retry = int(value)
            except ValueError: retry = None
    if data: yield SSEvent(event=event, data="\n".join(data), id=event_id, retry=retry)


### `aiter_sse`

Async SSE parser from an httpx streamed response.


In [ ]:
#| export


async def aiter_sse(response: httpx.Response) -> AsyncIterator[SSEvent]:
    "Async SSE parser from an httpx streamed response."
    event, data, event_id, retry = None, [], None, None
    async for raw in response.aiter_lines():
        line = raw.rstrip("\n")
        if not line:
            if data: yield SSEvent(event=event, data="\n".join(data), id=event_id, retry=retry)
            event, data, event_id, retry = None, [], None, None
            continue
        if line.startswith(":"): continue
        field, _, value = line.partition(":")
        if value.startswith(" "): value = value[1:]
        if field == "event": event = value
        elif field == "data": data.append(value)
        elif field == "id": event_id = value
        elif field == "retry":
            try: retry = int(value)
            except ValueError: retry = None
    if data: yield SSEvent(event=event, data="\n".join(data), id=event_id, retry=retry)


## REPL Checks

Quick smoke checks to confirm exported symbols import correctly.


In [ ]:
#| hide
from fastcore.test import test
import fastllm_v2.sse as m

test(m is not None)
test(hasattr(m, 'SSEvent'))
